# Data Cleaning Silver Layer

In [9]:
# =============================================================================
# CREATE SILVER CATALOG & DATABASE
# =============================================================================
spark.sql("CREATE DATABASE IF NOT EXISTS credit_risk_assessment.silver")

DataFrame[]

In [ ]:

# =============================================================================
# 1. CUSTOMERS
# =============================================================================
customers_df = spark.table("credit_risk_assessment.bronze.customers")

customers_clean = customers_df.dropDuplicates()

customers_clean.write.format("delta").mode("overwrite").saveAsTable("credit_risk_assessment.silver.customers")
customers_clean.show(2)

In [11]:

# =============================================================================
# 2. LOANS
# =============================================================================
loans_df = spark.table("credit_risk_assessment.bronze.loans")

loans_clean = loans_df.dropDuplicates()
loans_clean = loans_clean.dropna(subset=["loan_id", "customer_id"])
loans_clean = loans_clean.fillna({"collateral_value": 0.0})
loans_clean = loans_clean.filter(F.col("loan_amount") > 0)
loans_clean = loans_clean.filter(F.col("interest_rate").between(0, 100))
loans_clean = loans_clean.filter(F.col("loan_term_months") > 0)
loans_clean = loans_clean.withColumn("loan_type", F.lower(F.trim(F.col("loan_type"))))

loans_clean.write.format("delta").mode("overwrite").saveAsTable("credit_risk_assessment.silver.loans")
loans_clean.show(2)
from pyspark.sql.functions import col, count, mean, mode

# Check for missing values
missing_values = loans_df.select([count(col(c)).alias(c) for c in loans_df.columns]).collect()[0].asDict()

# Fill missing values with mean for numeric columns and mode for categorical columns
numeric_cols = [field.name for field in loans_df.schema.fields if field.dataType.simpleString() in ['int', 'double', 'float']]
categorical_cols = [field.name for field in loans_df.schema.fields if field.dataType.simpleString() in ['string']]

for col_name in numeric_cols:
    mean_val = loans_df.select(mean(col(col_name))).collect()[0][0]
    loans_df = loans_df.fillna({col_name: mean_val})

for col_name in categorical_cols:
    mode_val = loans_df.groupBy(col_name).count().orderBy('count', ascending=False).first()[0]
    loans_df = loans_df.fillna({col_name: mode_val})

# Remove duplicate rows
loans_df = loans_df.dropDuplicates()

# Check and handle data type inconsistencies
for field in loans_df.schema.fields:
    if field.dataType.simpleString() == 'string':
        loans_df = loans_df.withColumn(field.name, col(field.name).cast('string'))
    elif field.dataType.simpleString() == 'int':
        loans_df = loans_df.withColumn(field.name, col(field.name).cast('int'))
    elif field.dataType.simpleString() == 'double':
        loans_df = loans_df.withColumn(field.name, col(field.name).cast('double'))
    elif field.dataType.simpleString() == 'float':
        loans_df = loans_df.withColumn(field.name, col(field.name).cast('float'))

+-------+-----------+-----------+---------+-------------+----------------+----------------+
|loan_id|customer_id|loan_amount|loan_type|interest_rate|loan_term_months|collateral_value|
+-------+-----------+-----------+---------+-------------+----------------+----------------+
|    184|        103|   276091.0| mortgage|         9.61|              60|        591880.0|
|    540|        315|   239390.0| personal|        11.47|              60|        442037.0|
+-------+-----------+-----------+---------+-------------+----------------+----------------+
only showing top 2 rows



In [23]:
# =============================================================================
# 3. PAYMENT TRANSACTIONS
# =============================================================================
payments_df = spark.table("credit_risk_assessment.bronze.payment_transactions")

payments_clean = payments_df.dropDuplicates()
payments_clean = payments_clean.dropna(subset=["payment_id", "loan_id"])
payments_clean = payments_clean.fillna({"days_past_due": 0})
payments_clean = payments_clean.filter(F.col("payment_amount") > 0)
payments_clean = payments_clean.filter(F.col("days_past_due") >= 0)
payments_clean = payments_clean.withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))

payments_clean.write.format("delta").mode("overwrite").saveAsTable("credit_risk_assessment.silver.payment_transactions")

payments_clean.show(2)

In [23]:

# =============================================================================
# 4. CUSTOMER FINANCIALS
# =============================================================================
financials_df = spark.table("credit_risk_assessment.bronze.customer_financials")
financials_clean = financials_df.dropDuplicates()
financials_clean.write.format("delta").mode("overwrite").saveAsTable("credit_risk_assessment.silver.customer_financials")

financials_df.show(2)


+-----------+----------+--------+--------+---------------+
|customer_id|total_debt|  equity| revenue|revenue_decline|
+-----------+----------+--------+--------+---------------+
|          1|   54494.0|161271.0|214625.0|              0|
|          2|  230650.0| 76978.0|248621.0|              0|
+-----------+----------+--------+--------+---------------+
only showing top 2 rows



In [16]:

# =============================================================================
# 5. MACRO MARKET
# =============================================================================
macro_df = spark.table("credit_risk_assessment.bronze.macro_market")

macro_clean = macro_df.dropDuplicates()
macro_clean = macro_clean.dropna(subset=["record_id", "date"])
macro_clean = macro_clean.fillna({"interest_rate": 0.0, "inflation_rate": 0.0, "unemployment_rate": 0.0})
macro_clean = macro_clean.filter(F.col("date") <= F.current_date())
macro_clean = macro_clean.filter(F.col("unemployment_rate").between(0, 100))
macro_clean = macro_clean.filter(F.col("inflation_rate").between(-10, 50))

macro_clean.write.format("delta").mode("overwrite").saveAsTable("credit_risk_assessment.silver.macro_market")
macro_clean.show(2)

+---------+----------+-------------+--------------+-----------------+
|record_id|      date|interest_rate|inflation_rate|unemployment_rate|
+---------+----------+-------------+--------------+-----------------+
|       19|2024-07-31|         2.12|          4.37|             3.93|
|       18|2024-06-30|         4.09|          1.38|             4.82|
+---------+----------+-------------+--------------+-----------------+
only showing top 2 rows

